# Counterfactual Explanation Training Loop

Generate counterfactual explanations for chest X-ray images using fine-tuned Stable Diffusion.
This notebook processes a dataset of images through DDIM inversion and guided decoding.


In [ ]:
import os
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision.models import densenet121, DenseNet121_Weights
from diffusers import AutoencoderKL, DDIMScheduler, UNet2DConditionModel

# ── Import from utils.py (same folder) ──────────────────────────────────────
# BinEmbedding: embedding module for probability bins
# prob_to_bin: convert classifier probability [0,1] → bin [0..9]
# tensor_to_pil: convert torch tensor → PIL image
# load_image: load image from path and apply transforms
# ddim_inversion: forward diffusion process (noise injection)
# cfg_decode: reverse diffusion with classifier-free guidance
from utils import BinEmbedding, prob_to_bin, tensor_to_pil, load_image, ddim_inversion, cfg_decode

## Config

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR           = "/kaggle/input/cxr-data-set"
CHECKPOINT_DIR     = "/kaggle/working/cxr_sd_finetuned/checkpoint_epoch20"
PRETRAINED         = "stable-diffusion-v1-5/stable-diffusion-v1-5"
OUTPUT_DIR         = "/kaggle/working/counterfactuals"

IMAGE_SIZE         = 512
BATCH_SIZE         = 4
NUM_STEPS          = 50
GUIDANCE_SCALE     = 7.5
INVERSION_STRENGTH = 0.8

# Target bin: which probability bin to decode toward.
# bin 0 = p(PNEUMONIA) ≈ 0.0  →  looks healthy
# bin 9 = p(PNEUMONIA) ≈ 1.0  →  looks most pneumonic
TARGET_BIN         = 0

## Setup

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt   = Path(CHECKPOINT_DIR)

print(f"Device: {device}")
print(f"Loading from checkpoint: {ckpt}")

## Load Models

In [ ]:
# Classifier
classifier = densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1)
classifier.classifier = torch.nn.Linear(classifier.classifier.in_features, 1)
classifier = classifier.to(device).eval()
for param in classifier.parameters():
    param.requires_grad = False

# Diffusion models from pretrained checkpoint
vae       = AutoencoderKL.from_pretrained(PRETRAINED, subfolder="vae").to(device).eval()
unet      = UNet2DConditionModel.from_pretrained(ckpt / "unet").to(device).eval()
scheduler = DDIMScheduler.from_pretrained(PRETRAINED, subfolder="scheduler")
bin_emb   = BinEmbedding().to(device).eval()
bin_emb.load_state_dict(torch.load(ckpt / "bin_emb.pt", map_location=device))

# Freeze all models
for param in vae.parameters():
    param.requires_grad = False
for param in unet.parameters():
    param.requires_grad = False
for param in bin_emb.parameters():
    param.requires_grad = False

print("All models loaded and frozen!")

## Load Dataset

In [ ]:
from torchvision.transforms import Compose, Resize, ToTensor, Normalize

transforms = Compose([
    Resize((IMAGE_SIZE, IMAGE_SIZE)),
    ToTensor(),
    Normalize([0.5], [0.5])
])

dataset = ImageFolder(DATA_DIR, transform=transforms)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Dataset size: {len(dataset)}")
print(f"Classes: {dataset.classes}")

## Generate Counterfactuals - Main Loop

In [ ]:
@torch.no_grad()
def generate_counterfactual(pixel_values, target_bin_val):
    """
    Generate counterfactual explanation for a single image
    
    Args:
        pixel_values: (1, 3, H, W) image tensor
        target_bin_val: int, target probability bin
    
    Returns:
        dict with original, counterfactual, and difference map
    """
    # Get source bin from classifier
    logits = classifier(pixel_values).squeeze()
    source_prob = torch.sigmoid(logits)
    source_bin = prob_to_bin(source_prob.unsqueeze(0))
    
    # Prepare embeddings
    target_bin_t = torch.tensor([target_bin_val], device=device)
    null_bin_t   = torch.tensor([bin_emb.null_token], device=device)
    
    source_emb = bin_emb(source_bin)
    target_emb = bin_emb(target_bin_t)
    null_emb   = bin_emb(null_bin_t)
    
    # Encode to latent
    latent = vae.encode(pixel_values).latent_dist.sample() * vae.config.scaling_factor
    
    # DDIM Inversion & CFG Decoding
    inversion_step = int(INVERSION_STRENGTH * NUM_STEPS)
    trajectory = ddim_inversion(latent, unet, scheduler, source_emb, device, NUM_STEPS)
    z_counter = cfg_decode(trajectory[inversion_step], start_step=NUM_STEPS - inversion_step,
                           unet=unet, scheduler=scheduler, null_emb=null_emb, target_emb=target_emb,
                           guidance_scale=GUIDANCE_SCALE, device=device, num_steps=NUM_STEPS)
    
    # Decode counterfactual
    counterfactual = vae.decode(z_counter / vae.config.scaling_factor).sample
    
    # Compute difference map
    diff = (counterfactual - pixel_values).abs()
    diff_norm = diff / (diff.max() + 1e-8)
    
    return {
        'original': pixel_values,
        'counterfactual': counterfactual,
        'diff': diff_norm,
        'source_prob': source_prob.item(),
        'source_bin': source_bin.item()
    }


# Generate counterfactuals for dataset
results = []
for batch_idx, (pixel_values, labels) in enumerate(dataloader):
    pixel_values = pixel_values.to(device)
    
    for idx in range(pixel_values.size(0)):
        img = pixel_values[idx:idx+1]
        result = generate_counterfactual(img, TARGET_BIN)
        result['label'] = labels[idx].item()
        results.append(result)
        
        print(f"[{batch_idx * BATCH_SIZE + idx + 1}/{len(dataset)}] "
              f"p(PNEUMONIA)={result['source_prob']:.3f}  bin={result['source_bin']} → {TARGET_BIN}")

print(f"\nGenerated {len(results)} counterfactuals!")

## Visualize and Save Results

In [ ]:
# Save a selection of results
num_to_save = min(10, len(results))
for idx in range(num_to_save):
    result = results[idx]
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(tensor_to_pil(result['original']), cmap="gray")
    axes[0].set_title(f"Original\np={result['source_prob']:.2f}  bin={result['source_bin']}")
    axes[0].axis("off")
    
    axes[1].imshow(tensor_to_pil(result['counterfactual']), cmap="gray")
    axes[1].set_title(f"Counterfactual\ntarget bin={TARGET_BIN}")
    axes[1].axis("off")
    
    im = axes[2].imshow(result['diff'].squeeze(0).mean(0).cpu().numpy(), cmap="hot")
    axes[2].set_title("Difference Map")
    axes[2].axis("off")
    plt.colorbar(im, ax=axes[2], fraction=0.046)
    
    plt.suptitle(f"Explanation {idx+1} | strength={INVERSION_STRENGTH}  guidance={GUIDANCE_SCALE}")
    plt.tight_layout()
    
    save_path = f"{OUTPUT_DIR}/explanation_{idx:03d}_bin{result['source_bin']}_to_{TARGET_BIN}.png"
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Saved: {save_path}")

print(f"\nAll results saved to {OUTPUT_DIR}")